# Week 2 — FashionSDXLGenerator Demo

End-to-end demonstration of the `FashionSDXLGenerator` pipeline.

**Run order:** Execute cells top-to-bottom. GPU recommended but CPU works.

---

## 0. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add repo root to path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from week2.generator.sdxl_generator import (
    FashionSDXLGenerator,
    GenerationOutput,
    SIZE_PRESETS,
    SCHEDULER_MAP,
)
from week2.prompts.style_presets import list_presets, get_preset
from week2.prompts.negative_prompts import get_fashion_negative, format_negative
from week2.logging_setup import setup_logging

setup_logging(level='INFO')
print('✓ Imports OK')

## 1. Generator Info

In [ ]:
# Check hardware and show available options
gen = FashionSDXLGenerator(
    device      = 'auto',       # auto-detects CUDA / MPS / CPU
    torch_dtype = 'float16',    # use float32 for CPU
    output_dir  = ROOT / 'week2/outputs/generated',
)

info = gen.get_info()
print(f"Device      : {info['device']}")
print(f"VRAM        : {info['vram_gb']:.1f} GB")
print(f"Loaded      : {info['is_loaded']}")
print(f"Sizes       : {', '.join(list(SIZE_PRESETS.keys())[:4])} ...")
print(f"Schedulers  : {', '.join(list(SCHEDULER_MAP.keys())[:5])} ...")
print(f"Styles      : {', '.join(list_presets())}")

## 2. Load the Model

In [ ]:
# Load SDXL weights (downloads from HuggingFace on first run ~7 GB)
# Set HF_TOKEN env var if you have a private account

gen.load_model(
    low_vram_mode      = False,   # Set True for 8 GB GPUs
    sequential_offload = False,   # Set True for <6 GB GPUs
)
print(f'Model loaded on {gen.device}')

## 3. Single Image Generation

In [ ]:
from IPython.display import display

PROMPT = "A woman wearing an elegant deep red silk evening gown, fashion photography, studio lighting"
NEG    = format_negative(get_fashion_negative())

result = gen.generate_image(
    prompt               = PROMPT,
    negative_prompt      = NEG,
    size_preset          = 'portrait_1024',
    num_inference_steps  = 30,
    guidance_scale       = 7.5,
    seed                 = 42,
    scheduler            = 'euler',
)

print(result)
if result.success:
    display(result.first_image)

## 4. Save Output

In [ ]:
if result.success:
    paths = gen.save_output(
        result,
        fmt            = 'png',
        save_metadata  = True,
        filename_prefix= 'evening_gown',
    )
    print(f'Saved to: {paths[0]}')

## 5. Batch Generation with Style Presets

In [ ]:
# Build prompts with style preset tags applied
from week2.prompts.prompt_builder import PromptBuilder

builder = PromptBuilder()

batch_configs = [
    dict(subject='oversized graphic hoodie with neon print',   style='streetwear', gender='unisex'),
    dict(subject='tailored double-breasted navy blazer',        style='formal',     gender='men'),
    dict(subject='flowy floral maxi dress in pastel shades',   style='bohemian',   gender='women'),
]

prompts  = []
negatives= []
for cfg in batch_configs:
    built = builder.build(**cfg)
    prompts.append(built.positive)
    negatives.append(built.negative)

results = gen.generate_batch(
    prompts              = prompts,
    negative_prompts     = negatives,
    size_preset          = 'portrait_1024',
    num_inference_steps  = 25,
    guidance_scale       = 7.5,
)

for r, cfg in zip(results, batch_configs):
    status = '✓' if r.success else '✗'
    print(f"{status} {cfg['style']:<12} | seed={r.seed} | {r.generation_time_s:.1f}s")

In [ ]:
# Display all generated images
from IPython.display import display
for r in results:
    if r.success and r.first_image:
        display(r.first_image.resize((512, 512)))

## 6. Save Batch Outputs

In [ ]:
all_paths = []
for result, cfg in zip(results, batch_configs):
    if result.success:
        paths = gen.save_output(
            result,
            filename_prefix = cfg['style'],
            fmt             = 'png',
        )
        all_paths.extend(paths)

print(f'Saved {len(all_paths)} images')
for p in all_paths:
    print(f'  {p}')

## 7. Scheduler Comparison

In [ ]:
FIXED_PROMPT = "A minimalist white cotton shirt, clean studio background, fashion photography"
FIXED_NEG    = format_negative(get_fashion_negative())
FIXED_SEED   = 1234

schedulers_to_test = ['euler', 'dpm++', 'ddim']

sched_results = gen.generate_batch(
    prompts              = [FIXED_PROMPT] * len(schedulers_to_test),
    negative_prompts     = [FIXED_NEG] * len(schedulers_to_test),
    seeds                = [FIXED_SEED] * len(schedulers_to_test),
    num_inference_steps  = 20,
    size_preset          = 'square_512',
)

from IPython.display import display
for r, sched in zip(sched_results, schedulers_to_test):
    print(f'Scheduler: {sched} | {r.generation_time_s:.1f}s')
    if r.success and r.first_image:
        display(r.first_image)

## 8. Cleanup

In [ ]:
gen.unload_model()
print('Model unloaded. VRAM freed.')